# Metadata Filtering
- 벡터 유사도 계산 전에(또는 함께) 문서의 메타데이터 조건으로 검색 대상을 제한하는 기법

In [2]:
%pip install lark

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['COHERE_API_KEY'] = os.getenv('COHERE_API_KEY')

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [2]:
import pandas as pd

document_df = pd.read_csv('documents_meta.csv')
document_df

,doc_id,title,content,author,category
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...",김민수,여행
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기...",이영희,음식
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet...",박지훈,문화;음악
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...,최수정,역사;교육
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...,정우성,역사;군사
5,D6,2024년 기후 변화 종합 보고서,"2024년 전 지구 평균 기온은 산업화 이전 대비 약 1.2℃ 상승했으며, 해수면 ...",한예슬,환경;과학;보고서
6,D7,AI 기술 동향 및 윤리,"최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발...",강다니엘,기술;윤리;AI
7,D8,서울 지하철 이용 가이드,"서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로...",오정연,교통;여행
8,D9,판소리 “춘향가” 서사 구조,"판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가...",신동엽,문화;음악;역사
9,D10,한국 축구 대표팀 주요 기록,"한국 축구 대표팀은 2002 한일 월드컵 4강 진출, 2012 런던 올림픽 동메달 ...",윤아름,스포츠;역사


In [3]:
queries_df = pd.read_csv('queries_meta_v2.csv')
queries_df

,query_id,query_text,relevant_doc_ids
0,Q01,저자 김민수의 문서를 모두 보여줘,D1=1;D11=1;D21=1
1,Q02,저자 이영희의 문서를 모두 보여줘,D2=1;D12=1;D22=1
2,Q03,저자 박지훈의 문서를 모두 보여줘,D3=1;D13=1;D23=1
3,Q04,저자 최수정의 문서를 모두 보여줘,D4=1;D14=1;D24=1
4,Q05,저자 정우성의 문서를 모두 보여줘,D5=1;D15=1;D25=1
5,Q06,저자 한예슬의 문서를 모두 보여줘,D6=1;D16=1;D26=1
6,Q07,저자 강다니엘의 문서를 모두 보여줘,D7=1;D17=1;D27=1
7,Q08,저자 오정연의 문서를 모두 보여줘,D8=1;D18=1;D28=1
8,Q09,저자 신동엽의 문서를 모두 보여줘,D9=1;D19=1;D29=1
9,Q10,저자 윤아름의 문서를 모두 보여줘,D10=1;D20=1;D30=1
